<a href="https://colab.research.google.com/github/kul447/mysite/blob/main/currency_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Indian currency classifier -> INT8 for ESP32-S3
**How to use:** run the cells **from top to bottom, one at a time** (click the play button, wait for the green tick, then the next one).
Do **not** use the Gemini chat panel to edit this notebook, and do not restart the runtime in the middle (you would lose the files).

Before starting: *Runtime -> Change runtime type -> T4 GPU -> Save*.

## Cell 1 - check the setup
Shows the TensorFlow version and whether the GPU is on.

In [2]:
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "NONE -> Runtime > Change runtime type > T4 GPU")

TensorFlow 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Cell 2 - upload your photos and split them into train / val / test
Click **Choose files** and select your dataset **.zip** (you can select several zips at once, e.g. the Kaggle dataset + a zip of your own phone photos).

What the cell does:
1. Unzips everything.
2. Finds the class of each photo from its **folder name** (the folder name must contain 10, 20, 50, 100, 200 or 500; e.g. `10`, `Rs50`, `100_note`). Other folders (like 2000) are ignored.
3. Removes exact duplicate files and unreadable images, fixes phone-photo rotation, converts to RGB, shrinks big photos (speeds up training).
4. Splits each class randomly: **70% train** (model learns from these), **15% val** (used during training to check progress), **15% test** (the final exam - never used for learning).

In [3]:
import os, re, glob, shutil, random, hashlib, zipfile
from google.colab import files
from PIL import Image, ImageOps

CLASSES = ["10", "20", "50", "100", "200", "500"]
SPLIT = (0.70, 0.15, 0.15)
SEED = 42
EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
REUSE_UPLOADED_ZIPS = True

NAME_MAP = {"1hundred": "100", "2hundred": "200", "5hundred": "500",
            "onehundred": "100", "twohundred": "200", "fivehundred": "500",
            "twenty": "20", "fifty": "50", "ten": "10"}

for d in ("/content/raw", "/content/dataset"):
    shutil.rmtree(d, ignore_errors=True)
os.makedirs("/content/raw")

zips = sorted(glob.glob("/content/*.zip"))
if REUSE_UPLOADED_ZIPS and zips:
    print("Using zip file(s) already in Colab:", zips)
else:
    print("Click 'Choose files' and select your .zip file(s)...")
    up = files.upload()
    zips = [os.path.join("/content", n) for n in up if n.lower().endswith(".zip")]
for zp in zips:
    with zipfile.ZipFile(zp) as z:
        z.extractall(f"/content/raw/{os.path.splitext(os.path.basename(zp))[0]}")

def class_of(root):
    rel = os.path.relpath(root, "/content/raw").split(os.sep)[1:]
    for part in reversed(rel):
        for tok in re.findall(r"\d+", part):
            if tok in CLASSES:
                return tok
        key = re.sub(r"[^a-z0-9]", "", part.lower())
        for name in sorted(NAME_MAP, key=len, reverse=True):
            if name in key:
                return NAME_MAP[name]
    return None

found = {c: [] for c in CLASSES}
seen, dupes, ignored, mapping = set(), 0, {}, []
for root, dirs, fs in os.walk("/content/raw"):
    if "__MACOSX" in root:
        continue
    imgs = [f for f in fs if f.lower().endswith(EXT) and not f.startswith("._")]
    if not imgs:
        continue
    c = class_of(root)
    if c is None:
        ignored[root] = len(imgs); continue
    mapping.append((os.path.relpath(root, "/content/raw"), c, len(imgs)))
    for f in imgs:
        p = os.path.join(root, f)
        h = hashlib.md5(open(p, "rb").read()).hexdigest()
        if h in seen:
            dupes += 1; continue
        seen.add(h); found[c].append(p)

print("\nFolder -> note it was assigned to:")
for rel, c, n in sorted(mapping):
    print(f"  {rel:55s} -> Rs{c:<4} ({n} photos)")
print("\nFolders IGNORED (not one of the 6 notes):")
for rel, n in sorted(ignored.items()):
    print(f"  {os.path.relpath(rel, '/content/raw'):55s} ({n} photos)")

bad = 0
rng = random.Random(SEED)
for c in CLASSES:
    paths = sorted(found[c]); rng.shuffle(paths)
    n = len(paths); a = int(n * SPLIT[0]); b = a + int(n * SPLIT[1])
    for split, ps in {"train": paths[:a], "val": paths[a:b], "test": paths[b:]}.items():
        os.makedirs(f"/content/dataset/{split}/{c}", exist_ok=True)
        for k, p in enumerate(ps):
            try:
                with Image.open(p) as im:
                    im = ImageOps.exif_transpose(im).convert("RGB")
                    im.thumbnail((512, 512))
                    im.save(f"/content/dataset/{split}/{c}/{k}.jpg", "JPEG", quality=92)
            except Exception:
                bad += 1

print(f"\n{'note':>6} {'train':>6} {'val':>5} {'test':>5}")
for c in CLASSES:
    cnt = [len(os.listdir(f"/content/dataset/{s}/{c}")) for s in ("train", "val", "test")]
    print(f"{'Rs'+c:>6} {cnt[0]:>6} {cnt[1]:>5} {cnt[2]:>5}" + ("   <-- very few photos!" if sum(cnt) < 100 else ""))
print("\nexact duplicates removed:", dupes, "| unreadable images skipped:", bad)

Click 'Choose files' and select your .zip file(s)...


Saving data (1).zip to data (1).zip

Folder -> note it was assigned to:
  data (1)/Test/1Hundrednote                              -> Rs100  (6 photos)
  data (1)/Test/2Hundrednote                              -> Rs200  (6 photos)
  data (1)/Test/5Hundrednote                              -> Rs500  (6 photos)
  data (1)/Test/Fiftynote                                 -> Rs50   (6 photos)
  data (1)/Test/Tennote                                   -> Rs10   (6 photos)
  data (1)/Test/Twentynote                                -> Rs20   (6 photos)
  data (1)/Train/1Hundrednote                             -> Rs100  (22 photos)
  data (1)/Train/2Hundrednote                             -> Rs200  (22 photos)
  data (1)/Train/5Hundrednote                             -> Rs500  (22 photos)
  data (1)/Train/Fiftynote                                -> Rs50   (22 photos)
  data (1)/Train/Tennote                                  -> Rs10   (22 photos)
  data (1)/Train/Twentynote                           

## Cell 3 - train the model
Loads the three folders, then trains MobileNetV2 (alpha 0.35, 128x128). Watch `val_accuracy` go up.
Takes several minutes. Class weights are used automatically so a class with fewer photos is not ignored.

In [4]:
import numpy as np, tensorflow as tf
from tensorflow.keras import layers, models

IMG   = 128            # use 96 if the ESP32 runs out of memory
ARCH  = "mobilenetv2"  # or "dscnn"
BATCH, EPOCHS_HEAD, EPOCHS_FT, EPOCHS_DSCNN = 32, 8, 12, 40
tf.keras.utils.set_random_seed(42)
D = "/content/dataset"

kw = dict(class_names=CLASSES, label_mode="int", image_size=(IMG, IMG), batch_size=BATCH)
train_raw = tf.keras.utils.image_dataset_from_directory(f"{D}/train", shuffle=True, seed=42, **kw)
val_raw   = tf.keras.utils.image_dataset_from_directory(f"{D}/val",   shuffle=False, **kw)
test_raw  = tf.keras.utils.image_dataset_from_directory(f"{D}/test",  shuffle=False, **kw)

counts = np.array([len(os.listdir(f"{D}/train/{c}")) for c in CLASSES], dtype=float)
assert counts.min() > 0, "A class has 0 training photos - go back to Cell 2 and check the counts"
cw = {k: float(counts.sum() / (len(CLASSES) * counts[k])) for k in range(len(CLASSES))}
print("class weights:", {CLASSES[k]: round(v, 2) for k, v in cw.items()})

aug = tf.keras.Sequential([
    layers.RandomRotation(0.06), layers.RandomZoom(0.2), layers.RandomTranslation(0.06, 0.06),
    layers.RandomContrast(0.3), layers.RandomBrightness(0.3, value_range=(0, 255)),
])
train_ds = train_raw.map(lambda x, y: (tf.clip_by_value(aug(x, training=True), 0, 255), y),
                         num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds = val_raw.prefetch(tf.data.AUTOTUNE)

def build_mnv2():
    inp = layers.Input((IMG, IMG, 3))
    x = layers.Rescaling(1 / 127.5, offset=-1)(inp)          # scaling is INSIDE the model: feed raw 0..255
    base = tf.keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), alpha=0.35,
                                             include_top=False, weights="imagenet")
    base.trainable = False
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(len(CLASSES), activation="softmax")(x)
    return models.Model(inp, out), base

def build_dscnn():
    inp = layers.Input((IMG, IMG, 3))
    x = layers.Rescaling(1 / 127.5, offset=-1)(inp)
    x = layers.Conv2D(16, 3, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU(6.0)(x)
    for f in (32, 64, 128):
        x = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.ReLU(6.0)(x)
        x = layers.Conv2D(f, 1, use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.ReLU(6.0)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(len(CLASSES), activation="softmax")(x)
    return models.Model(inp, out), None

model, base = build_mnv2() if ARCH == "mobilenetv2" else build_dscnn()
loss = "sparse_categorical_crossentropy"

if ARCH == "mobilenetv2":
    model.compile(tf.keras.optimizers.Adam(1e-3), loss, metrics=["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=cw)
    base.trainable = True
    model.compile(tf.keras.optimizers.Adam(1e-4), loss, metrics=["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, class_weight=cw)
else:
    model.compile(tf.keras.optimizers.Adam(1e-3), loss, metrics=["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_DSCNN, class_weight=cw)

print("\nfloat32 val  acc:", round(model.evaluate(val_raw,  verbose=0)[1], 4))
print("float32 test acc:", round(model.evaluate(test_raw, verbose=0)[1], 4))

Found 103 files belonging to 6 classes.
Found 21 files belonging to 6 classes.
Found 29 files belonging to 6 classes.
class weights: {'10': 1.07, '20': 0.9, '50': 0.95, '100': 1.01, '200': 1.14, '500': 0.95}
2019640/2019640 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/8
4/4 ━━━━━━━━━━━━━━━━━━━━ 20s 3s/step - accuracy: 0.1748 - loss: 2.2444 - val_accuracy: 0.2381 - val_loss: 1.9141
Epoch 2/8
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 78ms/step - accuracy: 0.2524 - loss: 1.8422 - val_accuracy: 0.3810 - val_loss: 1.7783
Epoch 3/8
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 119ms/step - accuracy: 0.3883 - loss: 1.6286 - val_accuracy: 0.4762 - val_loss: 1.6622
Epoch 4/8
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 116ms/step - accuracy: 0.3883 - loss: 1.5842 - val_accuracy: 0.5238 - val_loss: 1.5488
Epoch 5/8
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 90ms/step - accuracy: 0.5437 - loss: 1.2237 - val_accuracy: 0.5238 - val_loss: 1.4658
Epoch 6/8
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 84ms/step - accuracy: 0.5437 - loss: 1.1449 - val_accuracy: 0.4762 - val_loss: 1.393

## Cell 4 - shrink to full INT8 and make the files
Converts the model to INT8 (weights **and** activations, plus int8 input/output) and writes `currency_model_int8.tflite` and `currency_model.h`.
**Copy the printed `INPUT scale, zero_point` line** - your firmware teammate needs it.

In [5]:
def rep_data():                      # ~320 real, un-augmented photos, raw 0..255
    for imgs, _ in train_raw.take(10):
        for i in range(imgs.shape[0]):
            yield [imgs[i:i + 1]]

conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep_data
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
tflite_model = conv.convert()

TFLITE = "currency_model_int8.tflite"
open(TFLITE, "wb").write(tflite_model)
print(f"{TFLITE}: {len(tflite_model)/1024:.1f} KB")

it = tf.lite.Interpreter(model_path=TFLITE); it.allocate_tensors()
inp, out = it.get_input_details()[0], it.get_output_details()[0]
assert inp["dtype"] == np.int8 and out["dtype"] == np.int8
s, z = inp["quantization"]
print("INPUT  scale, zero_point:", s, z, "   <- give to firmware teammate")
print("OUTPUT scale, zero_point:", *out["quantization"])
print("CLASS ORDER:", dict(enumerate(CLASSES)))

os.system(f"xxd -i {TFLITE} > currency_model.h")
NAME = "currency_model_int8_tflite"
if os.path.getsize("currency_model.h") == 0:          # fallback if xxd is missing
    d = open(TFLITE, "rb").read()
    body = ",\n".join("  " + ", ".join(f"0x{b:02x}" for b in d[i:i+12]) for i in range(0, len(d), 12))
    open("currency_model.h", "w").write(f"unsigned char {NAME}[] = {{\n{body}\n}};\nunsigned int {NAME}_len = {len(d)};\n")
h = open("currency_model.h").read()
h = h.replace(f"unsigned char {NAME}[] = {{", f"const unsigned char {NAME}[] __attribute__((aligned(16))) = {{")
h = h.replace(f"unsigned int {NAME}_len", f"const unsigned int {NAME}_len")
open("currency_model.h", "w").write(h)
print("wrote currency_model.h")

Saved artifact at '/tmp/tmpe5pu7v2p'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor_6')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  133545495790608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133545495791376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133545495791184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133545495791568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133545495790416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133545495790800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133545495792144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133545495791952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133545495790992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133545495792336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133545495793

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


currency_model_int8.tflite: 635.3 KB
INPUT  scale, zero_point: 1.0 -128    <- give to firmware teammate
OUTPUT scale, zero_point: 0.00390625 -128
CLASS ORDER: {0: '10', 1: '20', 2: '50', 3: '100', 4: '200', 5: '500'}


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


wrote currency_model.h


## Cell 4b - will it fit on the ESP32-S3? (16 MB flash, 8 MB PSRAM)
Prints how much **flash** the model needs and estimates the **RAM (tensor arena)** it needs while running.
The RAM number is an estimate of the largest set of intermediate results alive at the same time; the real value comes from `interpreter->arena_used_bytes()` on the board (usually a bit higher).

In [8]:
def estimate_arena_kb(path):
    ip = tf.lite.Interpreter(model_path=path, experimental_op_resolver_type=
         tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)   # full graph, no fused delegate
    ip.allocate_tensors()
    td = {t["index"]: t for t in ip.get_tensor_details()}
    ops = ip._get_ops_details()
    size = lambda i: int(np.prod(td[i]["shape"])) * np.dtype(td[i]["dtype"]).itemsize
    i0 = ip.get_input_details()[0]["index"]; outs = [o["index"] for o in ip.get_output_details()]
    birth, death = {i0: -1}, {}
    for k, op in enumerate(ops):
        for t in op["outputs"]: birth.setdefault(int(t), k)
        for t in op["inputs"]:
            if int(t) >= 0: death[int(t)] = max(death.get(int(t), -1), k)
    for t in outs: death[t] = len(ops)
    live = [t for t in birth if t in death]
    return max(sum(size(t) for t in live if birth[t] <= k <= death[t]) for k in range(-1, len(ops) + 1)) / 1024

try:
    flash_kb = os.path.getsize(TFLITE) / 1024
    ram_kb = estimate_arena_kb(TFLITE)
    print(f"Model in FLASH : {flash_kb:7.1f} KB   ({flash_kb/16384:.1%} of 16 MB)")
    print(f"Arena in RAM   : ~{ram_kb:6.1f} KB   ({ram_kb/8192:.1%} of 8 MB PSRAM)")
    print("-> fits easily in flash and PSRAM." if ram_kb < 4000 else "-> too big, use IMG = 96 or ARCH = 'dscnn'.")
    print("Internal SRAM is only 512 KB and is shared with Wi-Fi, stacks and camera buffers,")
    print("so put the arena in PSRAM unless this estimate is far below that.")
except Exception as e:
    print("Could not estimate (not a problem for training):", e)

Model in FLASH :   635.3 KB   (3.9% of 16 MB)
Arena in RAM   : ~ 240.0 KB   (2.9% of 8 MB PSRAM)
-> fits easily in flash and PSRAM.
Internal SRAM is only 512 KB and is shared with Wi-Fi, stacks and camera buffers,
so put the arena in PSRAM unless this estimate is far below that.


## Cell 5 - the honest exam: test the INT8 model on the TEST photos
These photos were never used for training or tuning. The confusion matrix: **rows = real note, columns = what the model said**. A good model has big numbers on the diagonal.

In [10]:
import pandas as pd

def predict_probs(x):                # x: float32 (IMG, IMG, 3), raw 0..255
    q = np.clip(np.round(x / s + z), -128, 127).astype(np.int8)[None]
    it.set_tensor(inp["index"], q); it.invoke()
    sc, zp = out["quantization"]
    return (it.get_tensor(out["index"])[0].astype(np.float32) - zp) * sc

y_true, y_pred = [], []
for x, y in test_raw.unbatch():
    y_pred.append(int(np.argmax(predict_probs(x.numpy())))); y_true.append(int(y))
y_true, y_pred = np.array(y_true), np.array(y_pred)

print("INT8 TEST accuracy:", round(float((y_true == y_pred).mean()), 4), f"({len(y_true)} photos)\n")
cm = tf.math.confusion_matrix(y_true, y_pred, num_classes=len(CLASSES)).numpy()
print(pd.DataFrame(cm, index=[f"true {c}" for c in CLASSES], columns=[f"pred {c}" for c in CLASSES]))
print()
for k, c in enumerate(CLASSES):
    n = cm[k].sum()
    print(f"Rs{c:>3}: {cm[k, k]}/{n} correct" + (f" ({cm[k, k]/n:.0%})" if n else ""))

INT8 TEST accuracy: 0.6552 (29 photos)

          pred 10  pred 20  pred 50  pred 100  pred 200  pred 500
true 10         3        0        2         0         0         0
true 20         0        5        0         0         0         0
true 50         0        0        5         0         0         0
true 100        0        0        4         1         0         0
true 200        1        0        0         0         3         0
true 500        0        0        3         0         0         2

Rs 10: 3/5 correct (60%)
Rs 20: 5/5 correct (100%)
Rs 50: 5/5 correct (100%)
Rs100: 1/5 correct (20%)
Rs200: 3/4 correct (75%)
Rs500: 2/5 correct (40%)


## Cell 6 - the real-world test: your OWN photos
Upload photos you took yourself (phone or the ESP32 camera). **Tip:** name them `100_a.jpg`, `500_b.jpg`, ... (note value, underscore, anything) and the cell will mark each one OK / WRONG for you.
This is the most important test: it shows whether the model works on photos that look like what the device will see.

In [11]:
up = files.upload()
ok = tot = 0
for name in up:
    try:
        with Image.open(name) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            x = tf.image.resize(np.array(im, dtype=np.float32), (IMG, IMG)).numpy()
    except Exception as e:
        print(name, "-> cannot read:", e); continue
    p = predict_probs(x)
    top = np.argsort(p)[::-1][:3]
    line = ", ".join(f"Rs{CLASSES[k]} {p[k]*100:.0f}%" for k in top)
    m = re.match(r"(\d+)[_\-\s]", name)
    verdict = ""
    if m and m.group(1) in CLASSES:
        tot += 1; good = CLASSES[top[0]] == m.group(1); ok += good
        verdict = "  OK" if good else "  WRONG"
    print(f"{name}: {line}{verdict}")
if tot: print(f"\n{ok}/{tot} correct on your own photos")

Saving data (1).zip to data (1) (1).zip
data (1) (1).zip -> cannot read: cannot identify image file 'data (1) (1).zip'


## Cell 7 - download the files
Downloads both files to your laptop. Allow multiple downloads if Chrome asks.

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
files.download("currency_model_int8.tflite")
files.download("currency_model.h")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>